<a href="https://colab.research.google.com/github/anilkumargangadhara09/GenAI-ML/blob/main/Nestle_HR_Assistant_genai_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nestlé HR Assistant (Course-End Project)

This notebook implements an AI-powered conversational assistant for Nestlé’s HR policy documents using LangChain, OpenAI, ChromaDB, and Gradio.

- Situation: Improve the operational efficiency of Nestlé's HR function by enabling fast, accurate Q&A over HR policy PDFs.
- Task: Build a chatbot that answers questions grounded in a provided policy PDF.
- Action: Load and split the PDF, embed chunks, store in Chroma, retrieve top passages, answer using GPT-3.5 Turbo, and expose a Gradio UI.
- Result: A self-contained, user-friendly RAG chatbot interface with an end-to-end workflow.

The notebook covers: environment setup, document processing, vector store creation, retrieval-augmented generation (RAG), prompting, and a Gradio interface.

In [1]:
# Install requirements - Simplified approach to avoid dependency conflicts
import subprocess
import sys

def safe_install():
    packages = [
        "langchain-openai",
        "langchain-community",
        "langchain-text-splitters",
        "gradio",
        "chromadb",
        "pypdf",
        "python-dotenv"
    ]

    print("🔧 Installing packages...")
    for pkg in packages:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
            print(f"✅ {pkg} installed")
        except:
            print(f"⚠️ {pkg} failed - try manual installation")

    print("\n🎉 Installation process completed!")
    print("💡 If you see any warnings, restart the kernel and run the next cell")

safe_install()

🔧 Installing packages...
✅ langchain-openai installed
✅ langchain-community installed
✅ langchain-text-splitters installed
✅ gradio installed
✅ chromadb installed
✅ pypdf installed
✅ python-dotenv installed

🎉 Installation process completed!
💡 If you see any warnings, restart the kernel and run the next cell


In [2]:
import os
import logging
import hashlib
from pathlib import Path
from typing import Optional, List, Dict, Any
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Import LangChain components with error handling
try:
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import Chroma
    from langchain_core.prompts import PromptTemplate
    from langchain.chains import RetrievalQA
    logger.info("Successfully imported all LangChain components")
except ImportError as e:
    logger.error(f"Import error: {e}")
    print(f"❌ Import failed: {e}")
    print("💡 Try running the installation cell above first")
    raise

# Configuration class for better organization
class Config:
    """Configuration settings for the HR Assistant"""
    OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
    MODEL_NAME: str = os.getenv("MODEL_NAME", "gpt-3.5-turbo")
    TEMPERATURE: float = float(os.getenv("TEMPERATURE", "0.2"))
    CHUNK_SIZE: int = int(os.getenv("CHUNK_SIZE", "1000"))
    CHUNK_OVERLAP: int = int(os.getenv("CHUNK_OVERLAP", "200"))
    RETRIEVER_K: int = int(os.getenv("RETRIEVER_K", "4"))
    PERSIST_DIR: str = os.getenv("PERSIST_DIR", "./chroma_db_nestle")
    COLLECTION_NAME: str = os.getenv("COLLECTION_NAME", "nestle_hr")

    @classmethod
    def validate(cls) -> None:
        """Validate configuration settings"""
        if not cls.OPENAI_API_KEY:
            raise ValueError("❌ OPENAI_API_KEY environment variable is required. Please set it in your .env file or environment.")
        logger.info("✅ Configuration validated successfully")

ERROR:__main__:Import error: No module named 'langchain.chains'


❌ Import failed: No module named 'langchain.chains'
💡 Try running the installation cell above first


ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# Validate configuration and initialize components
Config.validate()

# File paths
PDF_PATH = Path("../work/nestle_hr_policy.pdf")

# Initialize OpenAI chat model
chat_model = ChatOpenAI(
    temperature=Config.TEMPERATURE,
    model_name=Config.MODEL_NAME,
    openai_api_key=Config.OPENAI_API_KEY
)

logger.info(f"Initialized ChatOpenAI model: {Config.MODEL_NAME}")

In [ ]:
class DocumentProcessor:
    """Handles document loading and processing operations"""

    def __init__(self, chunk_size: int = Config.CHUNK_SIZE, chunk_overlap: int = Config.CHUNK_OVERLAP):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
        logger.info(f"DocumentProcessor initialized with chunk_size={chunk_size}, overlap={chunk_overlap}")

    def load_and_split_pdf(self, pdf_path: Path) -> List[str]:
        """
        Load and split PDF document into text chunks

        Args:
            pdf_path: Path to the PDF file

        Returns:
            List of text chunks

        Raises:
            FileNotFoundError: If PDF file doesn't exist
            Exception: For other processing errors
        """
        try:
            if not pdf_path.exists():
                raise FileNotFoundError(f"PDF file not found: {pdf_path}")

            logger.info(f"Loading PDF: {pdf_path}")
            loader = PyPDFLoader(str(pdf_path))
            documents = loader.load()

            if not documents:
                raise ValueError("No content found in PDF")

            # Combine all pages
            full_text = "\\n".join([doc.page_content for doc in documents])
            logger.info(f"Loaded {len(documents)} pages, total characters: {len(full_text)}")

            # Split into chunks
            text_chunks = self.text_splitter.split_text(full_text)
            logger.info(f"Created {len(text_chunks)} text chunks")

            return text_chunks

        except Exception as e:
            logger.error(f"Error processing PDF {pdf_path}: {e}")
            raise

# Initialize document processor
doc_processor = DocumentProcessor()

In [ ]:
class VectorStoreManager:
    """Manages vector store operations with caching and persistence"""

    def __init__(self):
        self.embeddings = OpenAIEmbeddings(openai_api_key=Config.OPENAI_API_KEY)
        self.persist_dir = Path(Config.PERSIST_DIR)
        self.vectordb = None
        logger.info("VectorStoreManager initialized")

    def _get_content_hash(self, text_chunks: List[str]) -> str:
        """Generate hash for content to detect changes"""
        content = "".join(text_chunks)
        return hashlib.md5(content.encode()).hexdigest()

    def _load_existing_vectordb(self) -> Optional[Chroma]:
        """Load existing vector database if available"""
        try:
            if self.persist_dir.exists():
                vectordb = Chroma(
                    collection_name=Config.COLLECTION_NAME,
                    embedding_function=self.embeddings,
                    persist_directory=str(self.persist_dir)
                )
                # Test if the collection has data
                if vectordb._collection.count() > 0:
                    logger.info(f"Loaded existing vector database with {vectordb._collection.count()} documents")
                    return vectordb
        except Exception as e:
            logger.warning(f"Could not load existing vector database: {e}")
        return None

    def create_or_load_vectorstore(self, text_chunks: List[str], force_recreate: bool = False) -> Chroma:
        """
        Create or load vector store with intelligent caching

        Args:
            text_chunks: List of text chunks to embed
            force_recreate: Force recreation even if existing store found

        Returns:
            Chroma vector store instance
        """
        try:
            # Try to load existing vector store first
            if not force_recreate:
                existing_db = self._load_existing_vectordb()
                if existing_db:
                    self.vectordb = existing_db
                    return self.vectordb

            logger.info("Creating new vector store...")

            # Create directory if it doesn't exist
            self.persist_dir.mkdir(parents=True, exist_ok=True)

            # Create new vector store
            self.vectordb = Chroma.from_texts(
                texts=text_chunks,
                embedding=self.embeddings,
                collection_name=Config.COLLECTION_NAME,
                persist_directory=str(self.persist_dir)
            )

            # Persist the database
            self.vectordb.persist()
            logger.info(f"Vector store created and persisted with {len(text_chunks)} chunks")

            return self.vectordb

        except Exception as e:
            logger.error(f"Error creating vector store: {e}")
            raise

# Initialize vector store manager
vector_manager = VectorStoreManager()

# Load and process PDF
text_chunks = doc_processor.load_and_split_pdf(PDF_PATH)

# Create or load vector store
vectordb = vector_manager.create_or_load_vectorstore(text_chunks)

logger.info("Vector store setup completed successfully")

In [ ]:
# Verify vector store initialization
if vectordb is None:
    raise RuntimeError("Vector store initialization failed")

# Test vector store functionality
try:
    collection_count = vectordb._collection.count()
    logger.info(f"Vector store verified: {collection_count} documents indexed")
    print(f"✅ Vector store ready with {collection_count} documents")
except Exception as e:
    logger.error(f"Vector store verification failed: {e}")
    raise

In [ ]:
class RAGChain:
    """Retrieval-Augmented Generation chain for HR queries"""

    def __init__(self, vectordb: Chroma, llm: ChatOpenAI, k: int = Config.RETRIEVER_K):
        self.vectordb = vectordb
        self.llm = llm
        self.k = k

        # Create retriever
        self.retriever = vectordb.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k}
        )

        # Create QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=self.retriever,
            return_source_documents=True,
            verbose=False
        )

        logger.info(f"RAG chain initialized with k={k}")

    def query(self, question: str) -> Dict[str, Any]:
        """
        Process a query using RAG

        Args:
            question: User question

        Returns:
            Dictionary with answer and source documents
        """
        try:
            if not question.strip():
                return {"answer": "Please provide a valid question.", "sources": []}

            logger.info(f"Processing query: {question[:100]}...")
            result = self.qa_chain.invoke({"query": question})

            return {
                "answer": result.get("result", "No answer generated"),
                "sources": result.get("source_documents", [])
            }

        except Exception as e:
            logger.error(f"Error processing query: {e}")
            return {"answer": f"Error processing query: {str(e)}", "sources": []}

# Initialize RAG chain
rag_chain = RAGChain(vectordb, chat_model)
print("✅ RAG chain initialized successfully")

In [ ]:
# Enhanced prompt template for HR assistant
HR_ASSISTANT_PROMPT = """You are a helpful and knowledgeable Nestlé HR Assistant.

Your role is to provide accurate, professional, and helpful responses to HR-related questions based on the company's HR policies and best practices.

Guidelines:
- Provide clear, concise, and actionable answers
- Reference specific policy sections when applicable
- If information is not available in the provided context, clearly state this
- Maintain a professional and supportive tone
- Focus on practical guidance and next steps

Context: {context}

Question: {question}

Answer: """

# Create prompt template
prompt_template = PromptTemplate(
    template=HR_ASSISTANT_PROMPT,
    input_variables=["context", "question"]
)

# Test the prompt formatting
test_question = "What are the best practices for conducting interviews?"
test_context = "Sample HR policy context..."

formatted_prompt = prompt_template.format(
    context=test_context,
    question=test_question
)

print("✅ Enhanced prompt template created")
print("\\n--- Sample Formatted Prompt ---")
print(formatted_prompt[:300] + "...")

In [ ]:
# Test the RAG chain with a sample question
test_question = "What are the best practices for conducting interviews?"

print(f"Testing RAG chain with question: '{test_question}'")
print("-" * 50)

try:
    result = rag_chain.query(test_question)

    print("Answer:")
    print(result["answer"])
    print("\\n" + "="*50)

    if result["sources"]:
        print(f"\\nRetrieved {len(result['sources'])} source documents:")
        for i, doc in enumerate(result["sources"][:2], 1):  # Show first 2 sources
            print(f"\\nSource {i}:")
            print(doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content)

    print("\\n✅ RAG chain test completed successfully")

except Exception as e:
    print(f"❌ RAG chain test failed: {e}")
    logger.error(f"RAG chain test error: {e}")

In [ ]:
class HRChatbot:
    """Enhanced HR Chatbot with comprehensive error handling and logging"""

    def __init__(self, rag_chain: RAGChain):
        self.rag_chain = rag_chain
        self.conversation_history = []
        logger.info("HR Chatbot initialized")

    def _validate_input(self, user_query: str) -> str:
        """Validate and clean user input"""
        if not user_query or not isinstance(user_query, str):
            raise ValueError("Please enter a valid question")

        cleaned_query = user_query.strip()
        if len(cleaned_query) < 3:
            raise ValueError("Question is too short. Please provide more details")

        if len(cleaned_query) > 1000:
            raise ValueError("Question is too long. Please keep it under 1000 characters")

        return cleaned_query

    def _log_interaction(self, query: str, response: str, success: bool = True):
        """Log user interactions for monitoring"""
        interaction = {
            "query": query[:100],  # Truncate for privacy
            "response_length": len(response),
            "success": success,
            "timestamp": logger.handlers[0].formatter.formatTime(logging.LogRecord("", 0, "", 0, "", (), None))
        }
        self.conversation_history.append(interaction)
        logger.info(f"Interaction logged - Success: {success}, Query length: {len(query)}")

    def chat(self, user_query: str) -> str:
        """
        Main chatbot interface with comprehensive error handling

        Args:
            user_query: User's question

        Returns:
            Chatbot response
        """
        try:
            # Validate input
            cleaned_query = self._validate_input(user_query)

            # Process with RAG chain
            result = self.rag_chain.query(cleaned_query)
            response = result["answer"]

            # Add source information if available
            if result.get("sources"):
                source_count = len(result["sources"])
                response += f"\\n\\n📚 *Answer based on {source_count} relevant document(s) from HR policies.*"

            # Log successful interaction
            self._log_interaction(cleaned_query, response, success=True)

            return response

        except ValueError as e:
            error_msg = f"Input Error: {str(e)}"
            self._log_interaction(user_query or "", error_msg, success=False)
            return error_msg

        except Exception as e:
            error_msg = f"I apologize, but I encountered an error processing your question. Please try rephrasing or contact support if the issue persists."
            logger.error(f"Chatbot error: {e}")
            self._log_interaction(user_query or "", error_msg, success=False)
            return error_msg

    def get_stats(self) -> Dict[str, Any]:
        """Get chatbot usage statistics"""
        total_interactions = len(self.conversation_history)
        successful_interactions = sum(1 for i in self.conversation_history if i["success"])

        return {
            "total_interactions": total_interactions,
            "successful_interactions": successful_interactions,
            "success_rate": successful_interactions / total_interactions if total_interactions > 0 else 0
        }

# Initialize the chatbot
hr_chatbot = HRChatbot(rag_chain)
print("✅ Enhanced HR Chatbot initialized successfully")

In [ ]:
import gradio as gr
from datetime import datetime

def create_gradio_interface():
    """Create an enhanced Gradio interface with better UX"""

    def chatbot_wrapper(message, history):
        """Wrapper function for Gradio ChatInterface"""
        try:
            response = hr_chatbot.chat(message)
            return response
        except Exception as e:
            logger.error(f"Gradio wrapper error: {e}")
            return "I apologize, but I'm experiencing technical difficulties. Please try again."

    def get_example_questions():
        """Return example questions for users"""
        return [
            "What is the company's policy on remote work?",
            "How do I request time off?",
            "What are the performance review procedures?",
            "What benefits are available to employees?",
            "How do I report a workplace issue?"
        ]

    # Create the interface
    interface = gr.ChatInterface(
        fn=chatbot_wrapper,
        title="🏢 Nestlé HR Assistant",
        description="""
        **Welcome to the Nestlé HR Assistant!**

        I'm here to help you with HR-related questions based on company policies and procedures.

        **Tips for better results:**
        - Be specific in your questions
        - Ask about policies, procedures, benefits, or workplace guidelines
        - If you need immediate assistance, contact HR directly

        **Example questions:** """ + " | ".join(get_example_questions()[:3]),

        examples=get_example_questions(),

        theme=gr.themes.Soft(
            primary_hue="blue",
            secondary_hue="gray"
        ),

        retry_btn="🔄 Retry",
        undo_btn="↩️ Undo",
        clear_btn="🗑️ Clear Chat",

        chatbot=gr.Chatbot(
            height=500,
            placeholder="Start by asking an HR-related question...",
            show_label=False
        ),

        textbox=gr.Textbox(
            placeholder="Type your HR question here...",
            container=False,
            scale=7
        )
    )

    return interface

# Create and configure the interface
demo = create_gradio_interface()

# Add custom CSS for better styling
demo.css = """
.gradio-container {
    max-width: 800px !important;
    margin: auto !important;
}
.chat-message {
    padding: 10px !important;
    margin: 5px 0 !important;
}
"""

print("✅ Enhanced Gradio interface created")
print("🚀 Ready to launch HR Assistant!")

In [ ]:
# Launch the HR Assistant interface
if __name__ == "__main__":
    try:
        print("🚀 Launching Nestlé HR Assistant...")
        print(f"📊 System Status:")
        print(f"   - Vector Store: {vectordb._collection.count()} documents indexed")
        print(f"   - Model: {Config.MODEL_NAME}")
        print(f"   - Retriever K: {Config.RETRIEVER_K}")

        # Launch with optimized settings
        demo.launch(
            share=False,  # Set to True if you want to share publicly
            server_name="127.0.0.1",
            server_port=7860,
            show_error=True,
            quiet=False,
            debug=False
        )

    except Exception as e:
        logger.error(f"Failed to launch interface: {e}")
        print(f"❌ Launch failed: {e}")
        print("💡 Try checking your environment setup and dependencies")

In [ ]:
# Create .env file template for environment variables
env_template = """# Nestlé HR Assistant Environment Configuration
# Copy this to .env and fill in your actual values

# OpenAI API Configuration
OPENAI_API_KEY=your_openai_api_key_here

# Optional: Model Configuration
MODEL_NAME=gpt-3.5-turbo
TEMPERATURE=0.2

# Optional: Vector Store Configuration
CHUNK_SIZE=1000
CHUNK_OVERLAP=200
RETRIEVER_K=4

# Optional: Logging Configuration
LOG_LEVEL=INFO
"""

# Write environment template
with open('.env.template', 'w') as f:
    f.write(env_template)

print("✅ Created .env.template file")
print("📝 Please copy .env.template to .env and add your OpenAI API key")

# Display system information
print("\\n📋 System Requirements:")
print("- Python 3.8+")
print("- OpenAI API key")
print("- PDF file: ../work/nestle_hr_policy.pdf")
print("- Internet connection for OpenAI API calls")

# Display usage statistics if available
if 'hr_chatbot' in locals():
    stats = hr_chatbot.get_stats()
    print(f"\\n📊 Current Session Stats:")
    print(f"- Total interactions: {stats['total_interactions']}")
    print(f"- Success rate: {stats['success_rate']:.1%}")